<a href="https://colab.research.google.com/github/TomaRaul/Comparative_Study/blob/main/dinoVSresnet21_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision timm albumentations tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models.segmentation import deeplabv3_resnet50
import timm
import numpy as np
from tqdm import tqdm

In [ ]:
import torchvision.transforms.functional as TF
import random

def custom_transforms(image, mask):
    image = TF.resize(image, (518, 518))
    mask = TF.resize(mask, (518, 518), interpolation=transforms.InterpolationMode.NEAREST)

    if random.random() > 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    image = TF.to_tensor(image)
    image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    mask = torch.from_numpy(np.array(mask)).long()

    final_mask = torch.zeros_like(mask)
    final_mask[mask == 1] = 1

    return image, final_mask

class PetDataset(datasets.OxfordIIITPet):
    def __getitem__(self, index):
        img, mask = super().__getitem__(index)
        return custom_transforms(img, mask)

train_ds = PetDataset(root='data', split='trainval', target_types='segmentation', download=True)
val_ds   = PetDataset(root='data', split='test', target_types='segmentation', download=True)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)

print("Dataset configurat corect pentru segmentare semantică!")

In [ ]:
model_resnet = deeplabv3_resnet50(pretrained=True)
model_resnet.classifier[4] = nn.Conv2d(256, 2, 1) # 2 clase: fundal + animal

In [ ]:
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m', pretrained=True, features_only=True)

class DinoSeg(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.decoder = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 2, kernel_size=1) # Ieșire: 2 clase (Background, Pet)
        )

    def forward(self, x):
        feat = self.backbone(x)[-1]

        out = self.decoder(feat)

        return nn.functional.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)

model_dino = DinoSeg(backbone).to(device)
print(f"Model DINOv2 configurat cu succes pe dispozitivul: {device}")

In [ ]:
def train(model, loader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0

    for imgs, masks in tqdm(loader):
        imgs = imgs.to(device)
        masks = masks.squeeze(1).long().to(device)

        out = model(imgs)

        if isinstance(out, dict):
            preds = out['out']
        else:
            preds = out

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def evaluate(model, loader):
    model.eval()
    ious = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)

            masks = masks.squeeze(1).long().to(device)

            out = model(imgs)

            if isinstance(out, dict):
                preds = out['out']
            else:
                preds = out

            preds = torch.argmax(preds, dim=1)

            pred_fg  = (preds == 1)
            mask_fg  = (masks == 1)

            inter = (pred_fg & mask_fg).sum(dim=(1,2))
            union = (pred_fg | mask_fg).sum(dim=(1,2))

            iou = (inter + 1e-6) / (union + 1e-6)
            ious.extend(iou.cpu().numpy())

    return float(np.mean(ious))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_resnet.to(device)
model_dino.to(device)

EPOCHS = 10
UNFREEZE_EPOCH = 1

opt_resnet = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_resnet.parameters()),
    lr=1e-4, weight_decay=1e-4
)

opt_dino = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_dino.parameters()),
    lr=1e-4, weight_decay=1e-4
)

resnet_loss_hist = []
dino_loss_hist = []
resnet_iou_hist = []
dino_iou_hist = []

unfrozen = False

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch}")

    model_resnet.train()
    l_resnet = train(model_resnet, train_loader, opt_resnet)
    torch.cuda.empty_cache()

    model_dino.train()
    l_dino = train(model_dino, train_loader, opt_dino)
    torch.cuda.empty_cache()

    model_resnet.eval()
    model_dino.eval()

    iou_resnet = evaluate(model_resnet, val_loader)
    iou_dino = evaluate(model_dino, val_loader)

    resnet_loss_hist.append(l_resnet)
    dino_loss_hist.append(l_dino)
    resnet_iou_hist.append(iou_resnet)
    dino_iou_hist.append(iou_dino)

    print(f"ResNet Loss: {l_resnet:.4f} | IoU: {iou_resnet:.4f}")
    print(f"DINO   Loss: {l_dino:.4f} | IoU: {iou_dino:.4f}")

    if epoch == UNFREEZE_EPOCH and not unfrozen:
        unfrozen = True
        print(">> Unfreezing backbones")

        for p in model_resnet.backbone.parameters():
            p.requires_grad = True

        opt_resnet = torch.optim.AdamW([
            {'params': model_resnet.backbone.parameters(), 'lr': 1e-5},
            {'params': model_resnet.classifier.parameters(), 'lr': 1e-4}
        ], weight_decay=1e-4)

        for p in model_dino.backbone.parameters():
            p.requires_grad = True

        opt_dino = torch.optim.AdamW([
            {'params': model_dino.backbone.parameters(), 'lr': 1e-5},
            {'params': model_dino.decoder.parameters(),  'lr': 1e-4}
        ], weight_decay=1e-4)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(10)

plt.figure()
plt.plot(epochs, resnet_loss_hist, label='ResNet Loss')
plt.plot(epochs, dino_loss_hist, label='DINO Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

plt.figure()
plt.plot(epochs, resnet_iou_hist, label='ResNet IoU')
plt.plot(epochs, dino_iou_hist, label='DINO IoU')
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("IoU Comparison")
plt.legend()
plt.show()


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "epoch": list(range(1, EPOCHS + 1)),
    "resnet_loss": resnet_loss_hist,
    "dino_loss": dino_loss_hist,
    "resnet_iou": resnet_iou_hist,
    "dino_iou": dino_iou_hist
})

df.to_csv("training_results.csv", index=False)
print(" Datele au fost salvate în training_results.csv")

plt.savefig("loss_comparison.png", dpi=300)
plt.show()

plt.savefig("iou_comparison.png", dpi=300)
plt.show()

In [ ]:
# Salvare modele antrenate
torch.save(model_resnet.state_dict(), 'model_resnet_final.pth')
torch.save(model_dino.state_dict(), 'model_dino_final.pth')
print("Modelele au fost salvate cu succes!")
print(f" model_resnet_final.pth")
print(f" model_dino_final.pth")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch

def visualize_model_comparison(index=0):
    img_tensor, mask_truth = val_ds[index]

    model_resnet.eval()
    model_dino.eval()

    input_batch = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        out_res = model_resnet(input_batch)['out']
        pred_res = out_res.argmax(1).squeeze().cpu().numpy()

        out_dino = model_dino(input_batch)
        pred_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    display_img = img_tensor.permute(1, 2, 0).cpu().numpy()
    display_img = (display_img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    display_img = np.clip(display_img, 0, 1)

    plt.figure(figsize=(20, 5))

    titles = ['Imagine Originală', 'Masca Reală (GT)', 'Predicție ResNet50', 'Predicție DINOv2']
    imgs = [display_img, mask_truth.cpu().numpy(), pred_res, pred_dino]

    for i in range(4):
        plt.subplot(1, 4, i+1)
        cmap = 'viridis' if i > 0 else None
        plt.imshow(imgs[i], cmap=cmap)
        plt.title(titles[i])
        plt.axis('off')

    plt.tight_layout()
    plt.show()

visualize_model_comparison(index=20)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import os

transform_img = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_segmentation(image_path, model, model_name):
    if not os.path.exists(image_path):
        print(f"Imaginea {image_path} nu exista.")
        return

    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = model(input_tensor)
        preds = out['out'] if isinstance(out, dict) else out
        mask = preds.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

def compare_models(image_path):
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Original", "ResNet", "DINOv2"]
    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Toate functiile de vizualizare sunt gata!")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def predict_segmentation(image_path, model, model_name):
    img_pil = Image.open(image_path).convert('RGB')

    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = model(input_tensor)
        preds = out['out'] if isinstance(out, dict) else out
        mask = preds.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

def compare_models(image_path):
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Imagine Originala", "Predictie ResNet", "Predictie DINOv2"]

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Functiile de testare au fost definite cu succes!")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import os

transform_img = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_segmentation(image_path, model, model_name):
    if not os.path.exists(image_path):
        print(f"Imaginea {image_path} nu exista.")
        return

    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = model(input_tensor)
        preds = out['out'] if isinstance(out, dict) else out
        mask = preds.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

def compare_models(image_path):
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Original", "ResNet", "DINOv2"]
    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Toate functiile de vizualizare sunt gata!")

In [ ]:
import os

image_dir = '/content/data/oxford-iiit-pet/images'
test_images = [
    'Abyssinian_1.jpg',
    'american_bulldog_146.jpg',
    'Birman_173.jpg',
    'Bengal_100.jpg'
]

print("Testare modele pe imagini sample...\n")

for img_name in test_images:
    img_path = os.path.join(image_dir, img_name)

    if os.path.exists(img_path):
        print(f"\n{'='*60}")
        print(f" Testare pe: {img_name}")
        print(f"{'='*60}")

        print("\n ResNet:")
        predict_segmentation(img_path, model_resnet, "ResNet")

        print("\n DINO:")
        predict_segmentation(img_path, model_dino, "DINO")

        print("\n Comparație directă:")
        compare_models(img_path)
    else:
        print(f"Imaginea {img_name} nu a fost găsită")

print("\n Testare completă!")